### **Fundamentos de redes neuronales en PyTorch: Del Perceptrón al MLP**

Este cuaderno reorganiza material base de PyTorch para dejarlo alineado con la **semana 3** del curso:

- Perceptrón y MLP
- Funciones de activación
- Función de pérdida
- Retropropagación
- Puente desde vectores numéricos hacia una tarea elemental de texto

**Idea central:** primero entendemos tensores, gradientes y optimización, luego cerramos el puente hacia un modelo neuronal básico que más adelante podrá usarse sobre texto vectorizado.

#### **1. Importaciones y configuración**

In [ ]:
import math
import timeit
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline
torch.manual_seed(42)
np.random.seed(42)

#### **2. Tensores básicos en PyTorch**

Un tensor es la estructura fundamental de datos en PyTorch. Puede representar un escalar, un vector, una matriz o un tensor de orden superior.

In [ ]:
torch_scalar = torch.tensor(3.14)
torch_vector = torch.tensor([1, 2, 3, 4])
torch_matrix = torch.tensor([[1, 2], [3, 4]])
torch_tensor3d = torch.tensor(
    [
        [[1, 2], [3, 4]],
        [[5, 6], [7, 8]],
    ],
    dtype=torch.float32,
)

print("scalar shape:", torch_scalar.shape)
print("vector shape:", torch_vector.shape)
print("matrix shape:", torch_matrix.shape)
print("tensor3d shape:", torch_tensor3d.shape)

#### **3. NumPy - PyTorch**

En muchos flujos de trabajo de ML y NLP se pasa con frecuencia entre arreglos de NumPy y tensores de PyTorch.

In [ ]:
x_np = np.random.random((4, 4)).astype(np.float32)
x_pt = torch.tensor(x_np, dtype=torch.float32)

print("NumPy array:")
print(x_np)
print("\nPyTorch tensor:")
print(x_pt)
print("\ndtypes:", x_np.dtype, x_pt.dtype)

In [ ]:
b_np = (x_np > 0.5)
b_pt = (x_pt > 0.5)

print("Mascara booleana en NumPy:")
print(b_np)
print("\nMascara booleana en PyTorch:")
print(b_pt)

In [ ]:
print("Suma en NumPy:", np.sum(x_np))
print("Suma en PyTorch:", torch.sum(x_pt).item())
print("\nTranspuesta en NumPy:")
print(np.transpose(x_np))
print("\nTranspuesta en PyTorch:")
print(torch.transpose(x_pt, 0, 1))
print("\nTransponer tensor 3D de (2,2,2) intercambiando ejes 0 y 2:")
print(torch.transpose(torch_tensor3d, 0, 2).shape)

#### **4. CPU y GPU**

La comparación siguiente está protegida para que el cuaderno no falle si no hay CUDA disponible.

In [ ]:
x = torch.rand(2**8, 2**8)

time_cpu = timeit.timeit("x @ x", globals=globals(), number=5)
print(f"Tiempo aproximado en CPU: {time_cpu:.4f} s")

if torch.cuda.is_available():
    device = torch.device("cuda")
    x_gpu = x.to(device)
    torch.cuda.synchronize()
    time_gpu = timeit.timeit(
        "torch.cuda.synchronize(); x_gpu @ x_gpu; torch.cuda.synchronize()",
        globals=globals(),
        number=5,
    )
    print(f"Tiempo aproximado en GPU: {time_gpu:.4f} s")
else:
    print("CUDA no está disponible en este entorno, se omite la medición en GPU.")

#### **5. Mover objetos a un dispositivo**

Esta utilidad ayuda cuando trabajamos con tensores sueltos, listas, tuplas o diccionarios.

In [ ]:
def move_to(obj, device): #
    if torch.is_tensor(obj):
        return obj.to(device)
    if isinstance(obj, dict):
        return {k: move_to(v, device) for k, v in obj.items()}
    if isinstance(obj, list):
        return [move_to(v, device) for v in obj]
    if isinstance(obj, tuple):
        return tuple(move_to(v, device) for v in obj)
    return obj

sample_batch = {
    "features": torch.randn(2, 3),
    "labels": torch.tensor([0, 1]),
}
move_to(sample_batch, torch.device("cpu"))

#### **6. Derivadas y `autograd`**

Trabajaremos con la función:

$$
f(x) = (x - 2)^2
$$

Su derivada analítica es:

$$
f'(x) = 2x - 4
$$

PyTorch puede calcular esta derivada automáticamente mediante retropropagación.

In [ ]:
def f(x):
    return torch.pow(x - 2.0, 2)

def f_prime(x):
    return 2 * x - 4

x = torch.tensor([-3.5], requires_grad=True)#leaf tensor
print("Gradiente inicial:", x.grad)

value = f(x) # construir un grafo computacional
print("f(x):", value.item())

value.backward() #recorre ese grafo desde el final hacia atras
print("Gradiente calculado por autograd:", x.grad.item())
print("Gradiente analítico:", f_prime(torch.tensor([-3.5])).item())

#### **7. Descenso por gradiente manual**

Aquí actualizamos el parámetro de forma explícita usando la derivada calculada por `autograd`.

In [ ]:
x = torch.tensor([-3.5], requires_grad=True)
eta = 0.1
history_manual = []

for epoch in range(20):
    y = f(x)
    y.backward()#dy/dx y lo dejamos x.grad
    with torch.no_grad():
        x -= eta * x.grad
    history_manual.append((epoch, x.item(), y.item()))
    x.grad.zero_()

history_manual[:5], history_manual[-1]

In [ ]:
epochs = [e for e, _, _ in history_manual]
xs = [v for _, v, _ in history_manual]
losses = [l for _, _, l in history_manual]

plt.figure(figsize=(7, 4))
plt.plot(epochs, losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Descenso por gradiente manual sobre f(x)=(x-2)^2")
plt.show()

print(f"Valor final aproximado de x: {xs[-1]:.4f}")

#### **8. Optimización con `torch.optim`**

Ahora hacemos lo mismo, pero usando un optimizador real de PyTorch.

In [ ]:
x_param = nn.Parameter(torch.tensor([-3.5]), requires_grad=True)
optimizer = torch.optim.SGD([x_param], lr=eta)
history_opt = []

for epoch in range(20):
    optimizer.zero_grad()
    loss = f(x_param)
    loss.backward()
    optimizer.step()
    history_opt.append((epoch, x_param.item(), loss.item()))

history_opt[:5], history_opt[-1]

#### **9. Del modelo lineal al perceptrón**

Un perceptrón calcula:

$$
z = xW^T + b
$$

Para clasificación binaria, ese valor puede interpretarse como un **logit**.  
Si solo usamos una transformación lineal, el modelo es lineal.  
Si añadimos capas ocultas y activaciones, obtenemos un **MLP**.

In [ ]:
X_lin = torch.tensor(
    [
        [-2.0, -1.0],
        [-1.0, -1.5],
        [1.0, 1.0],
        [2.0, 1.5],
    ],
    dtype=torch.float32,
)
y_lin = torch.tensor([0, 0, 1, 1], dtype=torch.long)

perceptron = nn.Linear(2, 2)
logits = perceptron(X_lin)#CrossEntropyLoss

print("Forma de entrada:", X_lin.shape)
print("Forma de logits:", logits.shape)
print("Primeros logits:")
print(logits)

#### **10. Funciones de activación**

Sin activaciones no lineales, apilar varias capas lineales equivale a una sola transformación lineal.  
Por eso las activaciones son esenciales en un MLP.

In [ ]:
z = torch.linspace(-4, 4, 400)

plt.figure(figsize=(8, 5))
plt.plot(z.numpy(), z.numpy(), label="identity")
plt.plot(z.numpy(), torch.sigmoid(z).numpy(), label="sigmoid")
plt.plot(z.numpy(), torch.tanh(z).numpy(), label="tanh")
plt.plot(z.numpy(), F.relu(z).numpy(), label="ReLU")
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.legend()
plt.title("Comparación de funciones de activación")
plt.show()

#### **11. Funciones de pérdida**

- `MSELoss`: útil sobre todo en regresión.
- `CrossEntropyLoss`: estándar para clasificación multiclase con logits.

En semana 3 conviene entender por qué un problema de clasificación normalmente usa `CrossEntropyLoss`.

In [ ]:
pred_reg = torch.tensor([[2.5], [0.3], [1.2]], dtype=torch.float32)
true_reg = torch.tensor([[3.0], [0.0], [1.0]], dtype=torch.float32)
mse = nn.MSELoss()(pred_reg, true_reg)

logits_cls = torch.tensor([[2.0, -1.0], [0.1, 1.3], [1.2, -0.4]], dtype=torch.float32)
true_cls = torch.tensor([0, 1, 0], dtype=torch.long)
ce = nn.CrossEntropyLoss()(logits_cls, true_cls)

print("MSELoss:", mse.item())
print("CrossEntropyLoss:", ce.item())

#### **12. Retropropagación en un perceptrón**

Observa cómo `loss.backward()` llena los gradientes de `weight` y `bias`.

In [ ]:
torch.manual_seed(0)
perceptron = nn.Linear(2, 2)
criterion = nn.CrossEntropyLoss()

logits = perceptron(X_lin)
loss = criterion(logits, y_lin)

perceptron.zero_grad()
loss.backward()

print("Loss:", loss.item())
print("\nGradiente de weight:")
print(perceptron.weight.grad)
print("\nGradiente de bias:")
print(perceptron.bias.grad)

#### **13. De perceptrón a MLP**

Ahora añadimos una capa oculta y una activación no lineal.  
Esto ya es un **MLP**.

In [ ]:
mlp = nn.Sequential(
    nn.Linear(2, 8),
    nn.Tanh(),
    nn.Linear(8, 2),
)
#loss.backward()
logits_mlp = mlp(X_lin)
print("Logits del MLP:")
print(logits_mlp)

#### **14. Mini-ejemplo de entrenamiento de un MLP**

Usaremos el problema XOR para mostrar por qué una capa no lineal puede aprender relaciones que un modelo lineal simple no modela bien.

In [ ]:
X_xor = torch.tensor(
    [
        [0.0, 0.0],
        [0.0, 1.0],
        [1.0, 0.0],
        [1.0, 1.0],
    ],
    dtype=torch.float32,
)
y_xor = torch.tensor([0, 1, 1, 0], dtype=torch.long)

xor_mlp = nn.Sequential(
    nn.Linear(2, 8),
    nn.Tanh(),
    nn.Linear(8, 2),
)

criterion = nn.CrossEntropyLoss() #loss
optimizer = torch.optim.Adam(xor_mlp.parameters(), lr=0.05)

train_loss_hist = []
train_acc_hist = []

for epoch in range(100):
    optimizer.zero_grad()
    logits = xor_mlp(X_xor)
    loss = criterion(logits, y_xor)
    loss.backward() # retroprogación
    optimizer.step() #actualización

    preds = logits.argmax(dim=1)
    acc = (preds == y_xor).float().mean().item()

    train_loss_hist.append(loss.item())
    train_acc_hist.append(acc)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(train_loss_hist)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pérdida del MLP sobre XOR")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(train_acc_hist)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Exactitud del MLP sobre XOR")
plt.show()

final_preds = xor_mlp(X_xor).argmax(dim=1)
print("Predicciones finales:", final_preds.tolist())
print("Etiquetas reales:", y_xor.tolist())

#### **15. Puente hacia NLP: del texto al vector**

Una red neuronal no recibe texto sin procesar, primero necesitamos una representación numérica.
Aquí usamos una bolsa de palabras (**bag of words**) mínima.  


In [ ]:
toy_texts = [
    "me encanta este curso",
    "este laboratorio fue excelente",
    "odio perder puntos",
    "la tarea fue terrible",
]
toy_labels = torch.tensor([1, 1, 0, 0], dtype=torch.long)

def tokenize(text):
    return text.lower().split()

vocab = sorted({token for text in toy_texts for token in tokenize(text)})
stoi = {token: i for i, token in enumerate(vocab)}

def vectorize_bow(text, stoi):
    vec = torch.zeros(len(stoi), dtype=torch.float32)
    for token in tokenize(text):
        if token in stoi:
            vec[stoi[token]] += 1.0
    return vec

X_text = torch.stack([vectorize_bow(t, stoi) for t in toy_texts])

print("Vocabulario:", vocab)
print("Forma de la matriz texto->vector:", X_text.shape)
print(X_text)

In [ ]:
text_mlp = nn.Sequential(
    nn.Linear(len(vocab), 8),
    nn.ReLU(),
    nn.Linear(8, 2),
)

logits_text = text_mlp(X_text)
loss_text = nn.CrossEntropyLoss()(logits_text, toy_labels)

print("Logits del mini-modelo de texto:")
print(logits_text)
print("\nLoss inicial sobre el corpus simple:", loss_text.item())

#### **16. Preguntas de sustentación**

- ¿Qué diferencia conceptual hay entre perceptrón y MLP?
- ¿Qué hace exactamente `optimizer.step()`?
- ¿Qué ocurriría si quitamos la activación en la capa oculta?
- ¿Por qué `CrossEntropyLoss` recibe logits y no etiquetas one-hot?
- ¿Por qué el texto debe vectorizarse antes de entrar al modelo?
- ¿Qué limitaciones tiene la representación bag-of-words?

In [ ]:
## Tus respuestas